In [1]:
%load_ext autoreload
%autoreload 2

In [4]:
import numpy as np
import os, site
from datetime import datetime
from datetime import timedelta
import stonesoup

from stonesoup.models.transition.nonlinear import CTRV_STV as CTRV
from stonesoup.types.groundtruth import GroundTruthPath, GroundTruthState

from stonesoup.models.measurement.nonlinear import CartesianToBearingRange
from stonesoup.models.measurement.linear import LinearGaussian
from stonesoup.types.detection import Detection

In [5]:
dt=1/15

In [ ]:
start_time   = datetime.now().replace(microsecond=0)
current_time = start_time + timedelta(seconds=dt)
np.random.seed(1991)

transition_model = CTRV(linear_noise_coeff=0.1,turn_noise_coeff=0.1)
timesteps        = [current_time]
truth            = GroundTruthPath([GroundTruthState([0, 0, np.pi/4, 2, np.pi], timestamp=current_time)])

# %%
# Create the truth path
for k in range(1, 50):
    current_time += timedelta(seconds=dt)
    timesteps.append(current_time)
    truth.append(GroundTruthState(
        transition_model.function(truth[k-1], noise=True, time_interval=timedelta(seconds=dt)),
        timestamp=current_time))

# %%
# Plot the ground truth.

from stonesoup.plotter import AnimatedPlotterly
plotter = AnimatedPlotterly(timesteps, tail_length=0.3)
plotter.plot_ground_truths(truth, [0, 1])
plotter.fig

In [7]:
measurement_model = LinearGaussian(
    ndim_state=5,  # Number of state dimensions (position and velocity in 2D)
    mapping=(0, 1),  # Mapping measurement vector index to state index
    noise_covar=np.array([[0.01, 0],  # Covariance matrix for Gaussian PDF
                          [0, 0.01]])
    )

# %%
# Populate the measurement array
measurements = []
for state in truth:
    measurement = measurement_model.function(state, noise=True)
    measurements.append(Detection(measurement, timestamp=state.timestamp,
                                  measurement_model=measurement_model))

# %%
# Plot those measurements

plotter.plot_measurements(measurements, [0, 1])
plotter.fig

In [8]:
from stonesoup.predictor.particle import ParticlePredictor
from stonesoup.resampler.particle import ESSResampler
from stonesoup.updater.particle import ParticleUpdater


transition_model = CTRV(linear_noise_coeff=1,turn_noise_coeff=1)
predictor = ParticlePredictor(transition_model)
resampler = ESSResampler()
updater = ParticleUpdater(measurement_model, resampler)

from scipy.stats import multivariate_normal

from stonesoup.types.numeric import Probability  # Similar to a float type
from stonesoup.types.state import ParticleState
from stonesoup.types.array import StateVectors

number_particles = 10000

# Sample from the prior Gaussian distribution
samples = multivariate_normal.rvs(np.array([0, 0, 0, 0, 0]),
                                  np.diag([4, 4, 2*np.pi, 4,  2*np.pi]),
                                  size=number_particles)

# Create prior particle state.
prior = ParticleState(state_vector=StateVectors(samples.T),
                      weight=np.array([Probability(1/number_particles)]*number_particles),
                      timestamp=start_time)

In [9]:
from stonesoup.types.hypothesis import SingleHypothesis
from stonesoup.types.track import Track

track = Track()
for measurement in measurements:
    prediction = predictor.predict(prior, timestamp=measurement.timestamp)
    hypothesis = SingleHypothesis(prediction, measurement)
    post = updater.update(hypothesis)
    track.append(post)
    prior = track[-1]

# %%
# Plot the resulting track with the sample points at each iteration. Can also change 'plot_history'
# to True if wanted.

plotter.plot_tracks(track, [0, 1], particle=True, plot_history=False)
plotter.fig




: 

: 

In [9]:
plotter = AnimatedPlotterly(timesteps, tail_length=0.3)
plotter.plot_ground_truths(truth, [0, 1])
plotter.plot_track_headings(track, [0, 1, 3, 2], plot_history=True, velocity_scale=0.1)
plotter.fig